In [11]:
!pip install langchain langchain-ollama langchain-community chromadb tiktoken


In [12]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [13]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import OllamaEmbeddings
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [18]:
# --- 1. Load and split documents ---
loader = DirectoryLoader("data", loader_cls=TextLoader, glob="**/*.*")
docs = loader.load()

# Use smaller chunks to get more diverse retrieval
splitter = RecursiveCharacterTextSplitter(chunk_size=10000, chunk_overlap=50)
chunks = splitter.split_documents(docs)

# --- 2. Create embeddings + local vector store ---
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vectorstore = Chroma.from_documents(chunks, embedding=embeddings, persist_directory="chroma_store")

# --- 3. Build retriever with diversity settings ---
retriever = vectorstore.as_retriever(
    search_type="mmr",  # Maximum Marginal Relevance for diversity
    search_kwargs={
        "k": 1,  # Retrieve more documents
        "lambda_mult": 0.7,  # Balance between relevance and diversity (0.0 = max diversity, 1.0 = max relevance)
        "fetch_k": 10  # Fetch more candidates before MMR selection
    }
)
llm = OllamaLLM(model="gemma3:latest")

# --- 4. Create a custom prompt template with system instructions ---
prompt_template = """You are a coding assistant that has to retrieve files and then convert them automatically into a different language.

Context:
{context}

Question: {question}

Answer with the complete code snippet from the file."""

prompt = PromptTemplate(
    template=prompt_template, 
    input_variables=["context", "question"]
)

# --- 5. Create RAG chain using the modern approach ---
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# --- 6. Ask a technical question ---
query = """Hi I want to create an "create intus" script, but instead of the default ps1 file format I want it to java based. Provide me with the code. And also write a short documentation at the top of the new code file."""

result = rag_chain.invoke(query)

print("🧠 Answer:\n", result, "\n")
print("📚 Retrieved documents from:")
retrieved_docs = retriever.invoke(query)
for doc in retrieved_docs:
    print(" -", doc.metadata["source"])

🧠 Answer:
 ```java
/**
 * This script creates and correlates an Intus-Inplanning account using a REST API.
 * It handles authentication, user retrieval, and account creation based on specified configurations.
 *
 * Prerequisites:
 * 1. Ensure you have the necessary API credentials (clientId, clientSecret, BaseUrl) configured.
 * 2. The script assumes the existence of a correlation configuration (accountField, accountFieldValue).
 *
 * Configuration:
 * - clientId: The client ID for authentication.
 * - clientSecret: The client secret for authentication.
 * - BaseUrl: The base URL of the Intus-Inplanning API.
 * - accountField: The field name for the account correlation.
 * - accountFieldValue: The value for the account correlation.
 */

import java.net.URL;
import java.util.HashMap;
import java.util.Map;

public class IntusInplanningCreator {

    public static void main(String[] args) {
        // Configuration (replace with your actual values)
        String clientId = "YOUR_CLIENT_I

In [15]:
# Let's try a more specific question that should pull from multiple files
query2 = """I need to understand how to create visualizations for sales data. 
Show me the chart generation functions and explain how to create a dashboard with multiple plots."""

result2 = qa({"query": query2})

print("🎨 Visualization Answer:\n", result2["result"], "\n")
print("📚 Retrieved from:")
for doc in result2["source_documents"]:
    print(" -", doc.metadata["source"])

print("\n" + "="*50 + "\n")

# And another query for reports
query3 = """How do I generate executive summaries and detailed reports from analysis results? 
Show me the report generation code."""

result3 = qa({"query": query3})

print("📊 Report Generation Answer:\n", result3["result"], "\n")
print("📚 Retrieved from:")
for doc in result3["source_documents"]:
    print(" -", doc.metadata["source"])

NameError: name 'qa' is not defined